# Chapter 17 — Transformers & Attention  ·  *PyTorch Companion*
**Track:** ML from Scratch · California Housing dataset (8 features treated as a sequence of tokens)

> This is the PyTorch companion to [notebook.ipynb](notebook.ipynb). The NumPy sections
> (positional encoding, scaled dot-product attention, multi-head demo) are framework-agnostic
> and unchanged. Every Keras model/training cell is followed by a runnable **PyTorch
> equivalent** built with `nn.MultiheadAttention` and `nn.LayerNorm`.
>
> Key PyTorch-transformer gotchas called out below:
> - `nn.MultiheadAttention` defaults to `batch_first=False` — we set `batch_first=True` to
>   match the usual `(batch, seq, d_model)` layout.
> - `nn.MultiheadAttention(...).forward(q, k, v)` returns `(attn_output, attn_weights)` —
>   remember to keep the second element when you want to plot heatmaps.
> - The causal-mask convention: PyTorch expects a **float** mask where masked positions are
>   `-inf` (additive), or a **bool** mask where `True` means "mask out" — opposite of Keras.
>   Use `torch.nn.Transformer.generate_square_subsequent_mask(T)` to get it right.

## Core Idea

The LSTM (Ch.8) solved the vanishing gradient problem but still processes tokens **sequentially** — step 1 must complete before step 2. Transformers discard recurrence entirely and replace it with **attention**: a single parallel operation that lets every position directly compare itself to every other position at once.

```
RNN:         x₁ → x₂ → x₃ → ... → xT     (sequential bottleneck)
Transformer: [x₁, x₂, x₃, ..., xT]        (all positions in parallel)
                      ↓
             Multi-Head Attention: every position attends to every other
```

**The key equations:**

Scaled dot-product attention:

$$\text{Attention}(Q,K,V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V$$

Positional encoding (sinusoidal):

$$\text{PE}_{(pos,\,2i)} = \sin\!\left(\frac{pos}{10000^{2i/d}}\right), \quad \text{PE}_{(pos,\,2i+1)} = \cos\!\left(\frac{pos}{10000^{2i/d}}\right)$$

## Running Example

The real estate platform has 8 tabular features per district in the California Housing dataset. We treat each feature as a **token** — a sequence of length 8. This is architecturally unconventional but pedagogically perfect: no new dataset, and the resulting attention heatmap has an immediately interpretable meaning ("which feature is attending to which other feature?").

Feature tokens: `MedInc`, `HouseAge`, `AveRooms`, `AveBedrms`, `Population`, `AveOccup`, `Latitude`, `Longitude`

In [ ]:
# TODO: Implement this cell


## The Math — Part 1: Positional Encoding

Attention is **permutation-equivariant**: shuffle the tokens and the output shuffles identically. Without positional information, the model cannot distinguish "feature 0 is MedInc" from "feature 5 is AveOccup". We inject position by **adding** a fixed encoding matrix to the token embeddings.

In [ ]:
# TODO: Implement this cell


## The Math — Part 2: Scaled Dot-Product Attention

For each token, we project the input into three roles — **Query** (what am I looking for?), **Key** (what do I offer?), **Value** (what do I contribute if selected?). The attention weights are dot-product similarities between queries and keys, scaled by $\sqrt{d_k}$ to prevent softmax saturation.

In [ ]:
# TODO: Implement this cell


## The Killer Visual — Attention Heatmap

Each row is a **query token** (the feature asking "who should I attend to?"). Each column is a **key token** (the feature being attended to). High weight = the model learned a strong relationship between those two features.

In [ ]:
# TODO: Implement this cell


## Encoder vs Decoder — One Mask Difference

| | Encoder | Decoder |
|---|---|---|
| Mask | None — every position sees every other | Causal mask — position t sees only ≤ t |
| Trained for | Representation, classification, embeddings | Token generation, LLMs |
| Examples | BERT, embedding models | GPT-4, Llama, Claude |

Two lines of code swap the architecture.

In [ ]:
# TODO: Implement this cell


## Multi-Head Attention

One set of Q, K, V learns one relationship pattern. Run H heads in parallel, each with its own projections (dimension `d_k = d_model / H`), then concatenate and project back.

One head might track income-location correlations. Another might track occupancy-room-count interactions. Each head specialises independently.

In [ ]:
# TODO: Implement this cell


## Full Transformer Encoder in Keras

Now we wire together the full encoder: token projection → positional encoding → N encoder blocks (LayerNorm + Multi-Head Attention + residual + FFN + residual) → global average pool → regression head.

In [ ]:
# TODO: Implement this cell


### PyTorch equivalent — tabular transformer encoder

| Keras | PyTorch |
|---|---|
| `layers.MultiHeadAttention(num_heads=H, key_dim=d_model//H)` | `nn.MultiheadAttention(embed_dim=d_model, num_heads=H, batch_first=True)` (`key_dim` is implied by `embed_dim // num_heads`) |
| `layers.LayerNormalization(epsilon=1e-6)` | `nn.LayerNorm(d_model, eps=1e-6)` |
| `layers.GlobalAveragePooling1D()` | `x.mean(dim=1)` (mean over the sequence axis) |
| implicit residual via `layers.Add` | explicit `x = x + sublayer(x)` in `forward` |
| fixed sinusoidal PE from the NumPy cell | register as a buffer so it moves with `.to(device)` and isn't a learned parameter |

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


### PyTorch equivalent — training loop with EarlyStopping

In [ ]:
# TODO: Implement this cell


In [ ]:
# TODO: Implement this cell


### PyTorch equivalent — training curves

In [ ]:
# TODO: Implement this cell


## Parameter Count: LSTM vs Transformer

In [ ]:
# TODO: Implement this cell


### PyTorch equivalent — LSTM baseline param count

In [ ]:
# TODO: Implement this cell


## The Hyperparameter Dial

In [ ]:
# TODO: Implement this cell


### PyTorch equivalent — `d_model` sweep

In [ ]:
# TODO: Implement this cell


## Exercises

1. **Change the mask.** In the encoder/decoder side-by-side cell, modify the causal mask to instead mask out only the **diagonal** (a token cannot attend to itself). Re-run the heatmap. Does the model still make sense? When would this be useful?

2. **Increase heads.** Change `num_heads` from 4 to 8 in `build_tabular_transformer`. What happens to `key_dim`? What constraint does this impose on `d_model`? Try `d_model=24, num_heads=8` — does it error? Why?

3. **Remove positional encoding.** Comment out the `x = x + pe[...]` line. Retrain and compare val RMSE. Does it matter for tabular data? Why would it matter more for natural language?

## Bridge

    "Ch.18 built the transformer encoder from first principles — attention, positional encoding, residuals, LayerNorm, and the one-mask difference between BERT and GPT. This is the architecture that every embedding model, every LLM, and every foundation model runs on.\n",

The AI track's `RAGAndEmbeddings` note picks up exactly here: embedding models *are* transformer encoders trained with contrastive loss to produce sentence vectors you can compare by cosine similarity. The pooling step in those notes (mean-pool across token outputs) is exactly the `GlobalAveragePooling1D` layer at the end of this chapter's model.

> *One architecture. Three deployment patterns: encode for embeddings (BERT), generate autoregressively (GPT), or condition on a separate encoder (T5-style encoder-decoder).*